# 03 - Bio contre conventionnel, à catégorie égale

**Objectif 5** : le label bio s'accompagne-t-il d'une meilleure qualité nutritionnelle et de moins d'additifs ?

**Hypothèses testées**
1. Les produits bio contiennent moins d'additifs (le cahier des charges bio en autorise beaucoup moins).
2. Les produits bio sont moins souvent ultra-transformés (NOVA 4).
3. Les produits bio ont un meilleur Nutri-Score.
4. Les produits bio contiennent moins de sucre, de sel et de graisses saturées.

**Méthode**
- **Comparer à catégorie égale.** Le bio est sur-représenté dans des rayons naturellement bien notés (céréales, fruits et légumes). Une comparaison tous rayons confondus mesurerait la composition des rayons, pas l'effet du bio. On compare donc *à l'intérieur* de chaque sous-groupe alimentaire (biscuits, fromages, jus de fruits...).
- **Effectifs minimaux.** Un sous-groupe n'est retenu que s'il compte au moins 100 produits bio et 100 conventionnels pour l'indicateur testé.
- **Tests.** Mann-Whitney pour les valeurs numériques, chi² pour les classes (Nutri-Score en lettres, NOVA).
- **Tailles d'effet.** Corrélation rang-bisériale *r* (de -1 à 1 ; négative si les valeurs sont plus basses en bio) et V de Cramér.
- **Tests multiples.** Avec des dizaines de tests, certains seraient significatifs par hasard. Les p-valeurs sont corrigées par la méthode de Benjamini-Hochberg. Un écart n'est retenu que s'il est significatif après correction **et** non négligeable (|r| >= 0,1).
- **Synthèse globale par standardisation directe.** On recalcule les taux bio et conventionnel comme si les deux avaient la même répartition entre sous-groupes.

Prérequis : `uv run python -m nutrilys.data.clean`.

In [1]:
import pandas as pd

from nutrilys.analysis import bio
from nutrilys.analysis.nutrition import label_food_groups
from nutrilys.db import products_connection
from nutrilys.viz import charts
from nutrilys.viz.fmt import fmt_int, fmt_num, fmt_pct
from nutrilys.viz.theme import INK_MUTED, SERIES, save_figure, use_theme

use_theme()
con = products_connection()
df = bio.load_frame(con)
findings = []

BIO, CONV = SERIES[2], SERIES[0]
VERDICT_COLORS = {
    "Plus bas en bio": SERIES[2],
    "Pas d'écart notable": INK_MUTED,
    "Plus haut en bio": SERIES[1],
}
n_bio = int(df["is_organic"].sum())
print(f"{fmt_int(len(df))} produits rattachés à un sous-groupe, dont {fmt_int(n_bio)} bio")

536 481 produits rattachés à un sous-groupe, dont 78 011 bio


## 1. Pourquoi comparer à catégorie égale

La part de bio varie fortement d'un rayon à l'autre.

In [2]:
comp = label_food_groups(bio.composition(df))
top, bottom = comp.iloc[0], comp.iloc[-1]

fig = charts.ranked_bars(
    comp,
    "food_group_label",
    "share_organic",
    suffix=" %",
    value_format=".0f",
    title=f"Le bio pèse {fmt_pct(top['share_organic'])} du rayon "
    f"{top['food_group_label'].lower()}, {fmt_pct(bottom['share_organic'])} "
    f"du rayon {bottom['food_group_label'].lower()}",
    subtitle="Part des produits bio par groupe alimentaire",
    value_label="Part de produits bio (%)",
)
save_figure(fig, "bio_share_by_food_group")
fig.show()

## 2. Comparaison brute contre comparaison à catégorie égale

Pour chaque indicateur : à gauche, le taux brut (tous sous-groupes mélangés) ; à droite, le taux standardisé (même composition pour le bio et le conventionnel). L'écart entre les deux panneaux mesure le biais de composition.

In [3]:
std = bio.standardized_table(df)
ab = std.set_index("indicator").loc["is_ab"]

fig = charts.dumbbell_panels(
    std,
    "label",
    panels=[
        ("Comparaison brute", "crude_conv", "crude_bio"),
        ("À catégorie égale", "std_conv", "std_bio"),
    ],
    names=("Conventionnel", "Bio"),
    colors=(CONV, BIO),
    title=f"Nutri-Score A ou B : écart bio de {fmt_num(ab['crude_gap'], 0)} points en brut, "
    f"{fmt_num(ab['std_gap'], 0)} à catégorie égale",
    subtitle=f"Part des produits (%), {int(ab['strata'])} sous-groupes comparables",
)
save_figure(fig, "bio_crude_vs_standardized")
fig.show()

In [4]:
std_view = std[
    [
        "label",
        "strata",
        "n_bio",
        "n_conv",
        "crude_bio",
        "crude_conv",
        "crude_gap",
        "std_bio",
        "std_conv",
        "std_gap",
    ]
]
std_view.columns = [
    "Indicateur",
    "Sous-groupes",
    "Bio",
    "Conventionnel",
    "Bio brut (%)",
    "Conv. brut (%)",
    "Écart brut",
    "Bio standardisé (%)",
    "Conv. standardisé (%)",
    "Écart standardisé",
]
std_view.style.format(precision=1, thousands=" ", decimal=",").hide(axis="index")

Indicateur,Sous-groupes,Bio,Conventionnel,Bio brut (%),Conv. brut (%),Écart brut,Bio standardisé (%),Conv. standardisé (%),Écart standardisé
Nutri-Score A ou B,34,62 374,363 655,"35,1","22,3","12,8","27,6","23,3","4,3"
Nutri-Score D ou E,34,62 374,363 655,"42,2","57,5","-15,3","49,1","56,3","-7,3"
Ultra-transformé (NOVA 4),34,47 078,230 809,"26,8","60,0","-33,2","31,0","59,1","-28,1"
Au moins un additif,34,48 858,231 731,"24,1","57,9","-33,8","29,2","56,7","-27,5"


In [5]:
for _, row in std.iterrows():
    findings.append(
        f"{row['label']} : bio {fmt_pct(row['std_bio'])} contre {fmt_pct(row['std_conv'])} "
        f"à catégorie égale (écart brut {fmt_num(row['crude_gap'], 0)} points, "
        f"standardisé {fmt_num(row['std_gap'], 0)} points)."
    )

## 3. Les additifs, sous-groupe par sous-groupe

Chaque point est un sous-groupe. Abscisse : taille d'effet *r* (négative si les produits bio contiennent moins d'additifs). Couleur : verdict après correction pour tests multiples. Le survol affiche les moyennes et les effectifs.

In [6]:
cont = bio.compare_continuous(df)
cont["detail"] = (
    "Moyenne bio "
    + cont["mean_bio"].map(lambda v: fmt_num(v, 1))
    + ", conventionnel "
    + cont["mean_conv"].map(lambda v: fmt_num(v, 1))
    + "<br>n = "
    + cont["n_bio"].map(fmt_int)
    + " bio / "
    + cont["n_conv"].map(fmt_int)
)


def effect_figure(outcome: str, what: str):
    sub = cont[cont["outcome"] == outcome]
    lower = int((sub["verdict"] == "Plus bas en bio").sum())
    higher = int((sub["verdict"] == "Plus haut en bio").sum())
    title = (
        f"{what} : plus bas en bio dans {lower} sous-groupes sur {len(sub)}, "
        f"plus haut dans {higher}"
    )
    fig = charts.effect_dots(
        sub,
        "label",
        "effect",
        "verdict",
        VERDICT_COLORS,
        hover="detail",
        title=title,
        subtitle="Taille d'effet rang-bisériale par sous-groupe (négative : plus bas en bio)",
        x_title="r  (négatif : plus bas en bio, positif : plus haut en bio)",
    )
    return fig, lower, higher, len(sub)


fig, lower, higher, total = effect_figure("n_additives", "Additifs")
save_figure(fig, "bio_additives_by_subgroup")
findings.append(
    f"Additifs : moins nombreux en bio dans {lower} sous-groupes sur {total}, "
    f"plus nombreux dans {higher}."
)
fig.show()

## 4. Le score Nutri-Score, sous-groupe par sous-groupe

On compare le score numérique sous-jacent (plus il est bas, meilleur est le produit), plus fin que la lettre.

In [7]:
fig, lower, higher, total = effect_figure("nutriscore_score", "Score Nutri-Score")
save_figure(fig, "bio_nutriscore_by_subgroup")
findings.append(
    f"Score Nutri-Score : meilleur en bio dans {lower} sous-groupes sur {total}, "
    f"moins bon dans {higher}."
)
fig.show()

## 5. Vue d'ensemble des cinq indicateurs

Pour chaque indicateur, répartition des sous-groupes testés selon le verdict.

In [8]:
counts = bio.verdict_counts(cont).reindex(columns=list(VERDICT_COLORS), fill_value=0)
shares = (
    (counts.div(counts.sum(axis=1), axis=0) * 100)
    .reset_index()
    .melt(id_vars="outcome", var_name="verdict", value_name="share")
)
labels = {k: v[0] for k, v in bio.CONTINUOUS_OUTCOMES.items()}
lowest = (counts["Plus bas en bio"] / counts.sum(axis=1)).idxmax()

fig = charts.stacked_shares(
    shares,
    "outcome",
    "verdict",
    "share",
    VERDICT_COLORS,
    segment_order=list(VERDICT_COLORS),
    row_order=list(bio.CONTINUOUS_OUTCOMES),
    segment_labels={k: k for k in VERDICT_COLORS},
    row_labels=labels,
    title=f"{labels[lowest]} : l'indicateur où le bio se distingue le plus souvent",
    subtitle="Part des sous-groupes testés, par verdict (après correction de Benjamini-Hochberg)",
)
fig.update_xaxes(title_text="Part des sous-groupes testés (%)")
save_figure(fig, "bio_verdicts_overview")
fig.show()

In [9]:
counts.rename(index=labels).assign(Total=counts.sum(axis=1).values)

verdict,Plus bas en bio,Pas d'écart notable,Plus haut en bio,Total
outcome,,,,
Nombre d'additifs,33,1,0,34
Score Nutri-Score,24,8,2,34
Sel,21,15,0,36
Graisses saturées,14,20,2,36
Sucres,18,15,3,36


## 6. Classes NOVA et lettres Nutri-Score

Test du chi² dans chaque sous-groupe : la répartition des classes diffère-t-elle entre bio et conventionnel ?

In [10]:
grades = pd.concat(
    [bio.compare_grades(df, "nova_group"), bio.compare_grades(df, "nutriscore_grade")]
)
summary = grades.groupby("outcome").agg(
    tested=("verdict", "size"),
    different=("significant", "sum"),
    median_v=("effect", "median"),
)
summary.index = summary.index.map({"nova_group": "Groupe NOVA", "nutriscore_grade": "Nutri-Score"})
for name, row in summary.iterrows():
    findings.append(
        f"{name} : répartition différente entre bio et conventionnel dans "
        f"{int(row['different'])} sous-groupes sur {int(row['tested'])} "
        f"(V de Cramér médian {fmt_num(row['median_v'], 2)})."
    )
summary.rename(
    columns={
        "tested": "Sous-groupes testés",
        "different": "Écart notable",
        "median_v": "V de Cramér médian",
    }
)

,Sous-groupes testés,Écart notable,V de Cramér médian
outcome,,,
Groupe NOVA,34,31,0.258765
Nutri-Score,34,21,0.120325


## 7. Synthèse

Constats calculés à partir des données ci-dessus.

In [11]:
for line in findings:
    print("-", line)

- Nutri-Score A ou B : bio 28 % contre 23 % à catégorie égale (écart brut 13 points, standardisé 4 points).
- Nutri-Score D ou E : bio 49 % contre 56 % à catégorie égale (écart brut -15 points, standardisé -7 points).
- Ultra-transformé (NOVA 4) : bio 31 % contre 59 % à catégorie égale (écart brut -33 points, standardisé -28 points).
- Au moins un additif : bio 29 % contre 57 % à catégorie égale (écart brut -34 points, standardisé -27 points).
- Additifs : moins nombreux en bio dans 33 sous-groupes sur 34, plus nombreux dans 0.
- Score Nutri-Score : meilleur en bio dans 24 sous-groupes sur 34, moins bon dans 2.
- Groupe NOVA : répartition différente entre bio et conventionnel dans 31 sous-groupes sur 34 (V de Cramér médian 0,26).
- Nutri-Score : répartition différente entre bio et conventionnel dans 21 sous-groupes sur 34 (V de Cramér médian 0,12).


## 8. Limites

- **Label sous-déclaré** : un produit bio dont le label n'a pas été saisi sur Open Food Facts est compté comme conventionnel. Ce bruit rapproche artificiellement les deux groupes : les écarts mesurés sont plutôt sous-estimés.
- **Association, pas causalité** : le bio va souvent de pair avec d'autres choix (marques spécialisées, recettes plus simples). On mesure ce qui accompagne le label, pas l'effet du label lui-même.
- **Hétérogénéité résiduelle** : même un sous-groupe reste varié (les « produits laitiers » mêlent yaourts nature et desserts). Un écart peut encore refléter des produits différents à l'intérieur du sous-groupe.
- **Bio et nutrition** : le cahier des charges bio porte sur le mode de production et les additifs autorisés, pas sur la teneur en sucre, sel ou graisses. Une absence d'écart sur ces nutriments n'est donc pas surprenante.
- **Couverture** : seuls les sous-groupes avec au moins 100 produits de chaque côté sont testés ; les conclusions ne s'étendent pas aux autres.